# Milestone 4: Transformers - Fine-Tuning BERT/DistilBERT

This notebook covers the development of a state-of-the-art transformer-based text classifier.

## Objective
Replace the custom Keras embedding classifier from Milestone 3 with a pretrained Hugging Face transformer. We fine-tune **DistilBERT** for binary sentiment classification and perform a direct comparison against the custom embedding model on performance (Accuracy, Macro F1), parameters, and inference latency.

### Roadmap
1. **Environment Setup & Colab Compatibility**
2. **Data Preparation**: Loading same splits, filtering 3-star reviews, and creating binary sentiment labels
3. **Tokenization**: Using DistilBERT fast tokenizer
4. **Fine-Tuning**: Fine-tuning `distilbert-base-uncased` using the Hugging Face `Trainer` API (PyTorch backend, optimized for GPUs)
5. **Inference Latency Benchmarking**: Measure milliseconds per review for both models
6. **Direct Comparison**: Quantifying accuracy, F1, model size, and latency trade-offs

## 1. Setup, Environment Detection & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Installing compatible datasets library (<3.0.0) and other packages...")
    !pip install "datasets>=2.16.0,<3.0.0" transformers[torch] diffusers accelerate gradio -q 2>/dev/null

    # Create folders
    os.makedirs('src', exist_ok=True)
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/plots', exist_ok=True)


    # Write data.py directly to Colab disk for imports

    data_py_content = 'import os\nimport json\nimport pandas as pd\nimport numpy as np\nimport requests\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\nfrom datasets import load_dataset\nimport io\nfrom PIL import Image\nfrom tqdm import tqdm\n\ndef load_amazon_data(category="All_Beauty"):\n    """\n    Loads raw reviews and metadata from Hugging Face datasets for a given category.\n    """\n    print(f"Loading reviews for {category}...")\n    reviews_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_review_{category}", trust_remote_code=True)\n    reviews_df = pd.DataFrame(reviews_dataset[\'full\'])\n    \n    print(f"Loading metadata for {category}...")\n    meta_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_meta_{category}", split="full", trust_remote_code=True)\n    meta_df = pd.DataFrame(meta_dataset)\n    \n    return reviews_df, meta_df\n\ndef clean_data(reviews_df, meta_df):\n    """\n    Performs basic cleaning of reviews and metadata:\n    - Handles missing values\n    - Formats data types (e.g. price, ratings)\n    - Extracts list elements from metadata where necessary\n    """\n    print("Cleaning metadata...")\n    # Clean price: convert to float if possible, or NaN\n    def clean_price(x):\n        if pd.isna(x):\n            return np.nan\n        if isinstance(x, (int, float)):\n            return float(x)\n        # If it\'s a string, try to parse\n        x_str = str(x).replace(\'$\', \'\').replace(\',\', \'\').strip()\n        try:\n            return float(x_str)\n        except ValueError:\n            return np.nan\n            \n    meta_df[\'price_cleaned\'] = meta_df[\'price\'].apply(clean_price)\n    \n    # Process images: extract the first image URL if it is a list of dicts/strings\n    def extract_image_url(images_val):\n        if not images_val or pd.isna(images_val):\n            return None\n            \n        def first_string(val):\n            if isinstance(val, list):\n                if len(val) > 0:\n                    first = val[0]\n                    if isinstance(first, str):\n                        return first\n                    elif isinstance(first, dict) and \'large\' in first:\n                        return first[\'large\']\n            elif isinstance(val, str):\n                return val\n            return None\n\n        if isinstance(images_val, dict):\n            for key in [\'large\', \'hi_res\', \'thumb\']:\n                if key in images_val:\n                    res = first_string(images_val[key])\n                    if res:\n                        return res\n            return None\n            \n        if isinstance(images_val, (list, np.ndarray)):\n            return first_string(list(images_val))\n            \n        return None\n\n    meta_df[\'image_url\'] = meta_df[\'images\'].apply(extract_image_url)\n    \n    # Clean descriptions (joining list of descriptions to single string)\n    def clean_description(desc):\n        if isinstance(desc, list):\n            return " ".join([str(d) for d in desc])\n        if pd.isna(desc):\n            return ""\n        return str(desc)\n        \n    meta_df[\'description_cleaned\'] = meta_df[\'description\'].apply(clean_description)\n    \n    print("Cleaning reviews...")\n    # Convert ratings and helpful votes\n    reviews_df[\'rating\'] = pd.to_numeric(reviews_df[\'rating\'], errors=\'coerce\')\n    vote_col = \'helpful_vote\' if \'helpful_vote\' in reviews_df.columns else \'helpful_votes\'\n    if vote_col in reviews_df.columns:\n        reviews_df[\'helpful_votes\'] = pd.to_numeric(reviews_df[vote_col], errors=\'coerce\').fillna(0).astype(int)\n    else:\n        reviews_df[\'helpful_votes\'] = 0\n    reviews_df[\'text_cleaned\'] = reviews_df[\'text\'].fillna("")\n    reviews_df[\'title_cleaned\'] = reviews_df[\'title\'].fillna("")\n    \n    return reviews_df, meta_df\n\ndef split_by_product(reviews_df, meta_df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42):\n    """\n    Splits the dataset by product (parent_asin) to prevent review-level leakage.\n    Ensures that all reviews and metadata for a product fall in the same split.\n    """\n    print("Splitting data by product...")\n    # We use parent_asin as the product identifier. If not present, we use asin.\n    prod_col = \'parent_asin\' if (\'parent_asin\' in meta_df.columns and \'parent_asin\' in reviews_df.columns) else \'asin\'\n    \n    # Get unique product IDs that exist in BOTH reviews and metadata\n    unique_products = list(set(meta_df[prod_col].unique()) & set(reviews_df[prod_col].unique()))\n    \n    # Train, validation, test split on products\n    train_prods, test_val_prods = train_test_split(\n        unique_products, \n        train_size=train_ratio, \n        random_state=random_seed\n    )\n    \n    val_relative_ratio = val_ratio / (val_ratio + test_ratio)\n    val_prods, test_prods = train_test_split(\n        test_val_prods, \n        train_size=val_relative_ratio, \n        random_state=random_seed\n    )\n    \n    train_prods_set = set(train_prods)\n    val_prods_set = set(val_prods)\n    test_prods_set = set(test_prods)\n    \n    # Split the metadata\n    meta_train = meta_df[meta_df[prod_col].isin(train_prods_set)]\n    meta_val = meta_df[meta_df[prod_col].isin(val_prods_set)]\n    meta_test = meta_df[meta_df[prod_col].isin(test_prods_set)]\n    \n    # Split the reviews\n    reviews_train = reviews_df[reviews_df[prod_col].isin(train_prods_set)]\n    reviews_val = reviews_df[reviews_df[prod_col].isin(val_prods_set)]\n    reviews_test = reviews_df[reviews_df[prod_col].isin(test_prods_set)]\n    \n    print(f"Split Summary (Products): Train={len(meta_train)}, Val={len(meta_val)}, Test={len(meta_test)}")\n    print(f"Split Summary (Reviews): Train={len(reviews_train)}, Val={len(reviews_val)}, Test={len(reviews_test)}")\n    \n    return (reviews_train, meta_train), (reviews_val, meta_val), (reviews_test, meta_test)\n\ndef download_and_cache_images(meta_df, output_dir, max_images=5000):\n    """\n    Downloads and caches product images locally.\n    Returns a mapping of product ID (asin or parent_asin) to local file path.\n    """\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n    \n    # Filter products that have valid image URLs\n    valid_images = meta_df[meta_df[\'image_url\'].notna() & (meta_df[\'image_url\'] != "")]\n    \n    # Sample up to max_images\n    if len(valid_images) > max_images:\n        valid_images = valid_images.sample(n=max_images, random_state=42)\n        \n    print(f"Downloading {len(valid_images)} product images to {output_path}...")\n    \n    image_paths = {}\n    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}\n    \n    # Determine the ID column\n    id_col = \'asin\' if \'asin\' in meta_df.columns else (\'parent_asin\' if \'parent_asin\' in meta_df.columns else \'asin\')\n    \n    for _, row in tqdm(valid_images.iterrows(), total=len(valid_images)):\n        prod_id = row[id_col]\n        url = row[\'image_url\']\n        ext = Path(url).suffix if Path(url).suffix in [\'.jpg\', \'.jpeg\', \'.png\'] else \'.jpg\'\n        local_file = output_path / f"{prod_id}{ext}"\n        \n        # Check if already downloaded\n        if local_file.exists():\n            image_paths[prod_id] = str(local_file)\n            continue\n            \n        try:\n            response = requests.get(url, headers=headers, timeout=10)\n            if response.status_code == 200:\n                img = Image.open(io.BytesIO(response.content))\n                # Convert to RGB if needed (handles RGBA/CMYK/etc)\n                if img.mode != \'RGB\':\n                    img = img.convert(\'RGB\')\n                img.save(local_file, "JPEG")\n                image_paths[prod_id] = str(local_file)\n            else:\n                image_paths[prod_id] = None\n        except Exception as e:\n            image_paths[prod_id] = None\n            \n    return image_paths\n'

    with open('src/data.py', 'w', encoding='utf-8') as f:

        f.write(data_py_content)

    sys.path.append(os.path.abspath('src'))

    data_dir_path = 'data/processed'
    plots_dir_path = 'data/plots'
else:
    sys.path.append(os.path.abspath('../src'))
    data_dir_path = '../data/processed'
    plots_dir_path = '../data/plots'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch

# HF datasets and transformers
import datasets
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Sklearn metrics
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Running in Google Colab. Installing compatible datasets library (<3.0.0) and other packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.0 MB/s eta 0:00:00
PyTorch Version: 2.11.0+cu128
CUDA Available: True


## 2. Load Processed Parquet Data & Prepare Sentiment Labels
We prepare the exact same binary sentiment split (Ratings 1,2 -> Negative (0), 4,5 -> Positive (1), excluding 3) to enable a mathematically rigorous comparison.

In [ ]:
from data import load_amazon_data, clean_data, split_by_product

# Check if processed splits exist
train_meta_path = Path(data_dir_path) / "meta_train.parquet"

if not train_meta_path.exists():
    print("Processed dataset splits not found. Loading and generating splits from scratch...")
    reviews_df, meta_df = load_amazon_data("All_Beauty")
    reviews_df, meta_df = clean_data(reviews_df, meta_df)

    splits = split_by_product(reviews_df, meta_df)
    (reviews_train, meta_train), (reviews_val, meta_val), (reviews_test, meta_test) = splits

    # Save splits
    os.makedirs(data_dir_path, exist_ok=True)
    reviews_train.to_parquet(Path(data_dir_path) / "reviews_train.parquet", index=False)
    reviews_val.to_parquet(Path(data_dir_path) / "reviews_val.parquet", index=False)
    reviews_test.to_parquet(Path(data_dir_path) / "reviews_test.parquet", index=False)

    meta_train.to_parquet(Path(data_dir_path) / "meta_train.parquet", index=False)
    meta_val.to_parquet(Path(data_dir_path) / "meta_val.parquet", index=False)
    meta_test.to_parquet(Path(data_dir_path) / "meta_test.parquet", index=False)
else:
    print("Loading processed splits from disk...")
    reviews_train = pd.read_parquet(Path(data_dir_path) / "reviews_train.parquet")
    reviews_val = pd.read_parquet(Path(data_dir_path) / "reviews_val.parquet")
    reviews_test = pd.read_parquet(Path(data_dir_path) / "reviews_test.parquet")

def prepare_sentiment_df(df):
    df_filtered = df[df['rating'] != 3].copy()
    df_filtered['sentiment'] = df_filtered['rating'].apply(lambda x: 1 if x > 3 else 0)
    df_filtered = df_filtered[df_filtered['text_cleaned'] != ""]
    # Rename columns to match HF conventions
    return df_filtered[['text_cleaned', 'sentiment']].rename(columns={'text_cleaned': 'text', 'sentiment': 'label'})

train_sent = prepare_sentiment_df(reviews_train)
val_sent = prepare_sentiment_df(reviews_val)
test_sent = prepare_sentiment_df(reviews_test)

# Since transformer fine-tuning on CPUs can be slow, if we aren't using a GPU we sample down,
# but with Colab T4 GPU we can run on the full split or a large chunk (e.g. 5,000 train samples)
if not torch.cuda.is_available():
    print("No GPU detected. Sub-sampling data to 1000 train, 200 test to run quickly on CPU...")
    train_sent = train_sent.sample(n=min(1000, len(train_sent)), random_state=42)
    test_sent = test_sent.sample(n=min(200, len(test_sent)), random_state=42)
    val_sent = val_sent.sample(n=min(200, len(val_sent)), random_state=42)

print(f"Train size: {len(train_sent)}, Val size: {len(val_sent)}, Test size: {len(test_sent)}")

Processed dataset splits not found. Loading and generating splits from scratch...
Loading reviews for All_Beauty...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating full split: 0 examples [00:00, ? examples/s]

Loading metadata for All_Beauty...


Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

Cleaning metadata...
Cleaning reviews...
Splitting data by product...
Split Summary (Products): Train=78795, Val=16885, Test=16885
Split Summary (Reviews): Train=485803, Val=108943, Test=106782
Train size: 446817, Val size: 99992, Test size: 98309


## 3. Hugging Face Datasets & Tokenization
We convert dataframes to Hugging Face `Dataset` objects and tokenize reviews using the DistilBERT uncased tokenizer.

In [ ]:
# Convert to HF datasets
train_dataset = Dataset.from_pandas(train_sent.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_sent.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_sent.reset_index(drop=True))

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Tokenization completed!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/446817 [00:00<?, ? examples/s]

Map:   0%|          | 0/99992 [00:00<?, ? examples/s]

Map:   0%|          | 0/98309 [00:00<?, ? examples/s]

Tokenization completed!


## 4. Fine-Tuning DistilBERT
We load the sequence classification model and set up the `Trainer` arguments.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU is active
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning...")
trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.094495,0.103145,0.963757,0.949239
2,0.102143,0.105296,0.965297,0.951718


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=27928, training_loss=0.0953936432142785, metrics={'train_runtime': 2958.4612, 'train_samples_per_second': 302.06, 'train_steps_per_second': 9.44, 'total_flos': 2.9594342832638976e+16, 'train_loss': 0.0953936432142785, 'epoch': 2.0})

## 5. Evaluation on Test Set

In [ ]:
print("Evaluating DistilBERT on test set...")
predictions = trainer.predict(tokenized_test)
logits = predictions.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = tokenized_test["label"]

transformer_acc = accuracy_score(y_true, y_pred)
transformer_macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"DistilBERT Test Accuracy: {transformer_acc:.4f}")
print(f"DistilBERT Test Macro F1: {transformer_macro_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

Evaluating DistilBERT on test set...


DistilBERT Test Accuracy: 0.9676
DistilBERT Test Macro F1: 0.9529

              precision    recall  f1-score   support

    Negative       0.93      0.93      0.93     21634
    Positive       0.98      0.98      0.98     76675

    accuracy                           0.97     98309
   macro avg       0.95      0.95      0.95     98309
weighted avg       0.97      0.97      0.97     98309



## 6. Benchmarking Latency & Performance Trade-offs
We measure the average time required to classify a single review on CPU/GPU and compare the trade-offs between custom embeddings and the DistilBERT Transformer.

In [ ]:
# Benchmark Transformer inference latency
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

sample_texts = test_sent['text'].head(100).tolist()

start_time = time.time()
for t in sample_texts:
    inputs = tokenizer(t, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        _ = model(**inputs)
transformer_latency_ms = ((time.time() - start_time) / len(sample_texts)) * 1000
print(f"DistilBERT Average Inference Latency: {transformer_latency_ms:.2f} ms per review")

DistilBERT Average Inference Latency: 7.77 ms per review


### 6.1 Direct Comparison Report

Fill in the metrics of your Keras Embeddings model from Milestone 3 below to complete your capstone report comparison:

In [ ]:
# Let's assume baseline embedding model values from a typical run for structure,
# you can plug in your exact numbers from Milestone 3.
embed_acc = 0.8250 # replace with exact value
embed_f1 = 0.8120 # replace with exact value
embed_latency_ms = 0.85 # average keras inference latency on CPU (approximate)

report_df = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test Macro F1', 'Latency (ms/review)', 'Model Parameters'],
    'Keras Learned Embeddings': [embed_acc, embed_f1, f"{embed_latency_ms:.2f} ms", "~650k (64-dim)"],
    'Fine-Tuned DistilBERT': [transformer_acc, transformer_macro_f1, f"{transformer_latency_ms:.2f} ms", "~66 Million"]
})

print("=== CAPSTONE COMPARISON REPORT ===")
print(report_df.to_string(index=False))

=== CAPSTONE COMPARISON REPORT ===
             Metric Keras Learned Embeddings Fine-Tuned DistilBERT
      Test Accuracy                    0.825              0.967602
      Test Macro F1                    0.812              0.952864
Latency (ms/review)                  0.85 ms               7.77 ms
   Model Parameters           ~650k (64-dim)           ~66 Million
